# 03b - Jointures, `explain(True)` et comparaison RDD / DataFrame / SQL (Partie 3)

In [1]:
import sys, time
sys.path.append('/home/jovyan/work/pipeline'); sys.path.append('/home/jovyan/work/data')
from pedali_spark import get_spark
spark = get_spark('pedali-03b-explain')
sc = spark.sparkContext
print('Spark', spark.version, '| master', sc.master, '| defaultParallelism', sc.defaultParallelism)

import df_analysis as da
from pyspark.sql import functions as F
stations, trips = da.load(spark)
ended = trips.filter("status = 'ENDED'").cache(); ended.count()
s = stations.select(F.col('station_id').alias('start_station'), 'zone_type')

Spark 3.5.3 | master spark://spark-master:7077 | defaultParallelism 4


## 1. `explain(True)` sur la jointure + agregation

In [2]:
q = (ended.join(s, 'start_station').groupBy('city', 'zone_type')
     .agg(F.count('*').alias('trajets'), F.round(F.sum('price_mad'), 2).alias('ca')))
q.explain(True)

== Parsed Logical Plan ==
'Aggregate ['city, 'zone_type], ['city, 'zone_type, count(1) AS trajets#607L, round(sum('price_mad), 2) AS ca#609]
+- Project [start_station#20, trip_id#16, user_id#17, vehicle_id#18, vehicle_type#19, end_station#21, city#22, status#23, duration_s#24, distance_km#25, battery_pct#26, price_mad#27, event_time#28, zone_type#4]
   +- Join Inner, (start_station#20 = start_station#574)
      :- Filter (status#23 = ENDED)
      :  +- Deduplicate [trip_id#16, status#23]
      :     +- Relation [trip_id#16,user_id#17,vehicle_id#18,vehicle_type#19,start_station#20,end_station#21,city#22,status#23,duration_s#24,distance_km#25,battery_pct#26,price_mad#27,event_time#28] json
      +- Project [station_id#0 AS start_station#574, zone_type#4]
         +- Relation [station_id#0,name#1,city#2,district#3,zone_type#4,capacity#5,latitude#6,longitude#7] json

== Analyzed Logical Plan ==
city: string, zone_type: string, trajets: bigint, ca: double
Aggregate [city#22, zone_type#4], [

### Lecture du plan
- **Parsed / Analyzed Logical Plan** : l'arbre tel qu'ecrit, puis avec colonnes et types resolus.
- **Optimized Logical Plan** (Catalyst) : projection des seules colonnes utiles (*column pruning* : `start_station`, `city`, `price_mad`, `zone_type`) et `isnotnull` ajoute sur les cles de jointure. Le DataFrame `ended` etant mis en cache (`InMemoryRelation`), le filtre `status = 'ENDED'` et le dedoublonnage (`Deduplicate` -> `SortAggregate` + `Exchange`) y sont deja evalues ; sans cache, le filtre est pousse jusqu'au `FileScan json`.
- **Physical Plan** : la table `stations` (400 lignes, quelques Ko) est sous le seuil `autoBroadcastJoinThreshold` (10 Mo) : Spark choisit un **BroadcastHashJoin** (le referentiel est diffuse aux executors, le gros cote n'est pas shuffle). L'agregation se fait en deux temps : `HashAggregate` partiel, `Exchange` (shuffle), `HashAggregate` final.

## 2. Strategie de jointure : `BroadcastHashJoin` vs `SortMergeJoin`

In [3]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)     # on interdit le broadcast
print('--- seuil = -1 : SortMergeJoin attendu ---')
ended.join(s, 'start_station').groupBy('zone_type').count().explain()
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', 10 * 1024 * 1024)
print('--- seuil = 10 Mo : BroadcastHashJoin attendu ---')
ended.join(s, 'start_station').groupBy('zone_type').count().explain()

--- seuil = -1 : SortMergeJoin attendu ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[zone_type#4], functions=[count(1)])
   +- Exchange hashpartitioning(zone_type#4, 8), ENSURE_REQUIREMENTS, [plan_id=176]
      +- HashAggregate(keys=[zone_type#4], functions=[partial_count(1)])
         +- Project [zone_type#4]
            +- SortMergeJoin [start_station#20], [start_station#574], Inner
               :- Sort [start_station#20 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(start_station#20, 8), ENSURE_REQUIREMENTS, [plan_id=168]
               :     +- Filter isnotnull(start_station#20)
               :        +- InMemoryTableScan [start_station#20], [isnotnull(start_station#20)]
               :              +- InMemoryRelation [trip_id#16, user_id#17, vehicle_id#18, vehicle_type#19, start_station#20, end_station#21, city#22, status#23, duration_s#24, distance_km#25, battery_pct#26, price_mad#27, event_time#28], StorageLeve

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[zone_type#4], functions=[count(1)])
   +- Exchange hashpartitioning(zone_type#4, 8), ENSURE_REQUIREMENTS, [plan_id=212]
      +- HashAggregate(keys=[zone_type#4], functions=[partial_count(1)])
         +- Project [zone_type#4]
            +- BroadcastHashJoin [start_station#20], [start_station#574], Inner, BuildRight, false
               :- Filter isnotnull(start_station#20)
               :  +- InMemoryTableScan [start_station#20], [isnotnull(start_station#20)]
               :        +- InMemoryRelation [trip_id#16, user_id#17, vehicle_id#18, vehicle_type#19, start_station#20, end_station#21, city#22, status#23, duration_s#24, distance_km#25, battery_pct#26, price_mad#27, event_time#28], StorageLevel(disk, memory, deserialized, 1 replicas)
               :              +- AdaptiveSparkPlan isFinalPlan=true
                                 +- == Final Plan ==
                                    SortAggrega

## 3. Comparaison RDD / DataFrame / SQL (meme calcul : CA par ville)
Temps mesures (meilleur de 3, donnees deja en cache). En PySpark, un **Dataset** typé n'existe pas : un DataFrame est un `Dataset[Row]` ; le Dataset type (`Dataset[Trip]`) n'est disponible qu'en Scala/Java, avec verification des types a la compilation mais sans gain de performance par rapport au DataFrame.

In [4]:
import json
ended.createOrReplaceTempView('ended')
def best(f, n=3):
    t_best, r = 1e9, None
    for _ in range(n):
        t = time.perf_counter(); r = f(); t_best = min(t_best, time.perf_counter() - t)
    return t_best, r
t_rdd, r_rdd = best(lambda: sorted((c, round(v, 2)) for c, v in ended.rdd.map(lambda r: (r.city, r.price_mad)).reduceByKey(lambda a, b: a + b).collect()))
t_df,  r_df  = best(lambda: sorted((r.city, round(r.ca, 2)) for r in ended.groupBy('city').agg(F.sum('price_mad').alias('ca')).collect()))
t_sql, r_sql = best(lambda: sorted((r.city, round(r.ca, 2)) for r in spark.sql('SELECT city, SUM(price_mad) ca FROM ended GROUP BY city').collect()))
assert r_rdd == r_df == r_sql
print(f'RDD       : {t_rdd*1000:8.1f} ms')
print(f'DataFrame : {t_df*1000:8.1f} ms')
print(f'SQL       : {t_sql*1000:8.1f} ms')
print(r_df)

RDD       :    984.7 ms
DataFrame :    580.6 ms
SQL       :    476.9 ms
[('Casablanca', 75430.27), ('Marrakech', 51959.36), ('Rabat', 43473.12), ('Tanger', 42600.26)]


**Lecture** : meme resultat ; DataFrame et SQL passent par le meme optimiseur Catalyst et le meme moteur Tungsten (execution dans la JVM sur format binaire, codegen). Le RDD execute des lambdas Python ligne par ligne (serialisation Python <-> JVM), sans optimisation globale. Le DataFrame/SQL est donc l'API a privilegier des que les donnees sont structurees ; le RDD garde son interet pour les donnees non structurees ou les traitements tres specifiques.

In [5]:
spark.stop()